In [ ]:
# ============================================================
# Figure6 final version
# RLP3/RLP4-associated immune-metabolic contexture
# in GSE202069 anti-PD-1-treated HCC cohort
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, spearmanr
from sklearn.preprocessing import StandardScaler

FIG6_DIR = "/cityu/htcc/zliu-39/R_loop_HCC/06_Figure6_GSE202069_PD1_final"
os.makedirs(FIG6_DIR, exist_ok=True)

def save_fig(fig, name, width=5, height=4):
    fig.set_size_inches(width, height)
    fig.savefig(os.path.join(FIG6_DIR, f"{name}.pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG6_DIR, f"{name}.svg"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG6_DIR, f"{name}.png"), dpi=600, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", name)

def p_to_star(p):
    if pd.isna(p):
        return "NA"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

In [ ]:
# ============================================================
# 1. Author-confirmed response annotation
# ============================================================

pd1_response = pd.DataFrame({
    "sample_id": [
        "PT16","PT10","PT15","PT6","PT19","PT7","PT1",
        "PT18","PT17","PT5","PT2","PT11","PT8","ST8",
        "PT14","PT4","PT3"
    ],
    "Response": [
        "Responder","Responder","Responder","Responder","Responder","Responder","Responder",
        "Non-responder","Non-responder","Non-responder","Responder",
        "Non-responder","Non-responder","Non-responder",
        "Non-responder","Non-responder","Non-responder"
    ]
})

pd1_response["Response"] = pd.Categorical(
    pd1_response["Response"],
    categories=["Non-responder", "Responder"],
    ordered=True
)

pd1_response["response_binary"] = np.where(
    pd1_response["Response"] == "Responder",
    1,
    0
)

pd1_samples = pd1_response["sample_id"].tolist()

print(pd1_response["Response"].value_counts())

pd1_response.to_csv(
    os.path.join(FIG6_DIR, "Figure6_PD1_response_annotation.csv"),
    index=False
)

In [ ]:
# ============================================================
# 2. Build PD1 response dataset
# ============================================================

missing_expr = [s for s in pd1_samples if s not in expr_t.index]
missing_score = [s for s in pd1_samples if s not in score_df.index]

print("Missing in expr_t:", missing_expr)
print("Missing in score_df:", missing_score)

pd1_use = [s for s in pd1_samples if s in expr_t.index and s in score_df.index]

pd1_data = (
    score_df.loc[pd1_use, ["RLP3_score", "RLP4_score", "RLP3_RLP4_combined_score"]]
    .join(pd1_response.set_index("sample_id"), how="inner")
)

pd1_data = pd1_data.loc[pd1_use]

median_score = pd1_data["RLP3_RLP4_combined_score"].median()

pd1_data["RLP3_RLP4_group"] = np.where(
    pd1_data["RLP3_RLP4_combined_score"] >= median_score,
    "RLP3/RLP4-high",
    "RLP3/RLP4-low"
)

pd1_data["RLP3_RLP4_group"] = pd.Categorical(
    pd1_data["RLP3_RLP4_group"],
    categories=["RLP3/RLP4-low", "RLP3/RLP4-high"],
    ordered=True
)

pd1_data.to_csv(
    os.path.join(FIG6_DIR, "Figure6_PD1_RLP3_RLP4_scores.csv")
)

display(pd1_data)

In [ ]:
# ============================================================
# Figure6A
# Response cohort overview
# ============================================================

plot_df = (
    pd1_data["Response"]
    .value_counts()
    .rename_axis("Response")
    .reset_index(name="Count")
)

order_response = ["Non-responder", "Responder"]

plot_df["Response"] = pd.Categorical(
    plot_df["Response"],
    categories=order_response,
    ordered=True
)

plot_df = plot_df.sort_values("Response").reset_index(drop=True)

palette_response = {
    "Non-responder": "#4E79A7",
    "Responder": "#E15759"
}

fig, ax = plt.subplots(figsize=(3.8, 4.0))

sns.barplot(
    data=plot_df,
    x="Response",
    y="Count",
    order=order_response,
    palette=palette_response,
    edgecolor="black",
    linewidth=0.8,
    ax=ax
)

for i, row in plot_df.iterrows():
    ax.text(
        i,
        row["Count"] + 0.25,
        str(row["Count"]),
        ha="center",
        va="bottom",
        fontsize=12
    )

ax.set_xlabel("")
ax.set_ylabel("Number of samples")
ax.set_title(
    "Response-annotated anti-PD-1 HCC cohort",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(fig, "Figure6A_PD1_response_cohort_overview", 3.8, 4.0)

In [ ]:
# ============================================================
# 3. Immune-metabolic signatures
# ============================================================

immune_signatures = {
    "MHC_I": ["HLA-A", "HLA-B", "HLA-C", "B2M", "TAP1", "TAP2", "TAPBP", "NLRC5"],
    "MHC_II": ["HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1", "HLA-DQA1", "HLA-DQB1", "CD74", "CIITA"],
    "IFNG_Response": ["IFNG", "STAT1", "IRF1", "CXCL9", "CXCL10", "CXCL11", "GBP1", "GBP2", "IDO1"],
    "Checkpoint": ["CD274", "PDCD1LG2", "PDCD1", "CTLA4", "LAG3", "HAVCR2", "TIGIT", "VSIR", "LGALS9", "CD47"],
    "Tcell_Exhaustion": ["PDCD1", "LAG3", "HAVCR2", "TIGIT", "CTLA4", "TOX", "ENTPD1", "CXCL13"],
    "Cytotoxicity": ["CD8A", "CD8B", "GZMA", "GZMB", "GZMH", "PRF1", "GNLY", "NKG7", "IFNG"],
    "Treg": ["FOXP3", "IL2RA", "CTLA4", "TIGIT", "IKZF2", "TNFRSF18", "CCR8"],
    "Immunosuppression": ["TGFB1", "VEGFA", "IL10", "IDO1", "ENTPD1", "NT5E", "CD47", "LGALS9"],
    "TAM_M2": ["CD68", "CD163", "MRC1", "MSR1", "MARCO", "C1QA", "C1QB", "C1QC", "APOE"],
    "SPP1_TAM_axis": ["SPP1", "CD44", "ITGAV", "ITGB1", "ITGB5", "MMP9", "VEGFA", "LGALS3"],
    "Hypoxia": ["HIF1A", "VEGFA", "CA9", "SLC2A1", "LDHA", "ENO1", "PGK1", "BNIP3"],
    "EMT": ["VIM", "FN1", "SNAI1", "SNAI2", "ZEB1", "ZEB2", "TWIST1", "CDH2", "COL1A1"]
}

expr_pd1 = expr_t.loc[pd1_use].copy()

immune_score_pd1 = pd.DataFrame(index=expr_pd1.index)

for sig, genes in immune_signatures.items():
    matched = [g for g in genes if g in expr_pd1.columns]
    print(sig, len(matched), matched)

    if len(matched) >= 2:
        X = expr_pd1[matched].copy()
        Xz = pd.DataFrame(
            StandardScaler().fit_transform(X),
            index=X.index,
            columns=X.columns
        )
        immune_score_pd1[sig] = Xz.mean(axis=1)

immune_score_pd1 = immune_score_pd1.join(
    pd1_data[[
        "RLP3_RLP4_combined_score",
        "RLP3_RLP4_group",
        "Response",
        "response_binary"
    ]]
)

immune_score_pd1.to_csv(
    os.path.join(FIG6_DIR, "Figure6_PD1_immune_signature_scores.csv")
)

display(immune_score_pd1.head())

In [ ]:
# ============================================================
# Figure6B
# Immune feature differences between responders and non-responders
# effect size plot
# ============================================================

sig_cols = [s for s in immune_signatures.keys() if s in immune_score_pd1.columns]

effect_rows = []

for sig in sig_cols:
    tmp = immune_score_pd1[[sig, "Response"]].dropna()

    nr = tmp.loc[tmp["Response"] == "Non-responder", sig]
    r = tmp.loc[tmp["Response"] == "Responder", sig]

    stat, pval = mannwhitneyu(nr, r, alternative="two-sided")

    effect_rows.append({
        "Signature": sig,
        "NR_mean": nr.mean(),
        "R_mean": r.mean(),
        "NR_median": nr.median(),
        "R_median": r.median(),
        "Delta_R_minus_NR": r.mean() - nr.mean(),
        "P_value": pval,
        "minus_log10_p": -np.log10(pval + 1e-300),
        "Significance": p_to_star(pval)
    })

effect_df = pd.DataFrame(effect_rows)
effect_df = effect_df.sort_values("Delta_R_minus_NR", ascending=True)

effect_df.to_csv(
    os.path.join(FIG6_DIR, "Figure6B_response_immune_feature_effect_size.csv"),
    index=False
)

fig, ax = plt.subplots(figsize=(6.4, 5.2))

colors = effect_df["Delta_R_minus_NR"].map(
    lambda x: "#E15759" if x > 0 else "#4E79A7"
)

ax.hlines(
    y=effect_df["Signature"],
    xmin=0,
    xmax=effect_df["Delta_R_minus_NR"],
    color=colors,
    linewidth=2,
    alpha=0.85
)

ax.scatter(
    effect_df["Delta_R_minus_NR"],
    effect_df["Signature"],
    s=80 + effect_df["minus_log10_p"] * 60,
    color=colors,
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

ax.axvline(0, color="grey", linestyle="--", linewidth=1)

for i, row in effect_df.reset_index(drop=True).iterrows():
    x = row["Delta_R_minus_NR"]
    offset = 0.04 if x >= 0 else -0.04
    ha = "left" if x >= 0 else "right"

    ax.text(
        x + offset,
        i,
        f"{x:.2f} {row['Significance']}",
        va="center",
        ha=ha,
        fontsize=9
    )

ax.set_xlabel("Mean signature difference: Responder - Non-responder")
ax.set_ylabel("")
ax.set_title(
    "Immune features enriched in anti-PD-1 responders",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(fig, "Figure6B_immune_feature_effect_size_response", 6.4, 5.2)

In [ ]:
# ============================================================
# Figure6C
# RLP3/RLP4-associated immune-metabolic states
# ============================================================

corr_rows = []

for sig in sig_cols:
    tmp = immune_score_pd1[[sig, "RLP3_RLP4_combined_score"]].dropna()

    r_val, p_val = spearmanr(
        tmp["RLP3_RLP4_combined_score"],
        tmp[sig]
    )

    corr_rows.append({
        "Signature": sig,
        "Spearman_r": r_val,
        "P_value": p_val,
        "minus_log10_p": -np.log10(p_val + 1e-300),
        "Significance": p_to_star(p_val)
    })

corr_df = pd.DataFrame(corr_rows)
corr_df = corr_df.sort_values("Spearman_r", ascending=True)

corr_df.to_csv(
    os.path.join(FIG6_DIR, "Figure6C_RLP3_RLP4_immune_metabolic_correlation.csv"),
    index=False
)

fig, ax = plt.subplots(figsize=(6.4, 5.2))

colors = corr_df["Spearman_r"].map(
    lambda x: "#E15759" if x > 0 else "#4E79A7"
)

ax.barh(
    corr_df["Signature"],
    corr_df["Spearman_r"],
    color=colors,
    edgecolor="black",
    linewidth=0.4
)

ax.axvline(0, color="grey", linestyle="--", linewidth=1)

for i, row in corr_df.reset_index(drop=True).iterrows():
    x = row["Spearman_r"]
    offset = 0.025 if x >= 0 else -0.025
    ha = "left" if x >= 0 else "right"

    ax.text(
        x + offset,
        i,
        f"{x:.2f} {row['Significance']}",
        va="center",
        ha=ha,
        fontsize=9
    )

ax.set_xlim(-0.65, 0.65)
ax.set_xlabel("Spearman correlation with RLP3/RLP4 score")
ax.set_ylabel("")
ax.set_title(
    "RLP3/RLP4-associated immune-metabolic states",
    fontsize=13,
    fontweight="bold"
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

save_fig(fig, "Figure6C_RLP3_RLP4_associated_immune_metabolic_states", 6.4, 5.2)

In [ ]:
# ============================================================
# Figure6D
# Sample-level heatmap integrating response, RLP3/RLP4 score, and immune states
# ============================================================

heatmap_features = [
    "RLP3_RLP4_combined_score",
    "Hypoxia",
    "MHC_I",
    "MHC_II",
    "IFNG_Response",
    "Checkpoint",
    "Tcell_Exhaustion",
    "Cytotoxicity",
    "SPP1_TAM_axis",
    "Immunosuppression"
]

heatmap_features = [f for f in heatmap_features if f in immune_score_pd1.columns]

heat_df = immune_score_pd1[heatmap_features + ["Response", "RLP3_RLP4_group"]].copy()

# 按 Response 和 RLP3/RLP4 score 排序
heat_df = heat_df.sort_values(
    ["Response", "RLP3_RLP4_combined_score"],
    ascending=[True, True]
)

mat = heat_df[heatmap_features].copy()

# column-wise z-score
mat_z = pd.DataFrame(
    StandardScaler().fit_transform(mat),
    index=mat.index,
    columns=mat.columns
)

# 转置：行=features，列=samples
plot_mat = mat_z.T

# 颜色注释
response_colors = {
    "Non-responder": "#4E79A7",
    "Responder": "#E15759"
}

rloop_colors = {
    "RLP3/RLP4-low": "#A0CBE8",
    "RLP3/RLP4-high": "#FF9D9A"
}

col_colors = pd.DataFrame({
    "Response": heat_df["Response"].map(response_colors),
    "RLP3/RLP4 group": heat_df["RLP3_RLP4_group"].map(rloop_colors)
}, index=heat_df.index)

g = sns.clustermap(
    plot_mat,
    cmap="RdBu_r",
    center=0,
    col_cluster=False,
    row_cluster=False,
    col_colors=col_colors,
    linewidths=0.2,
    linecolor="white",
    figsize=(10.5, 5.2),
    cbar_kws={"label": "Z-score"}
)

g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_ylabel("")
g.ax_heatmap.set_title(
    "Sample-level immune-metabolic contexture in anti-PD-1-treated HCC",
    fontsize=13,
    fontweight="bold",
    pad=18
)

plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha="right", fontsize=8)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=10)

# 保存
g.savefig(os.path.join(FIG6_DIR, "Figure6D_sample_level_heatmap.pdf"), bbox_inches="tight")
g.savefig(os.path.join(FIG6_DIR, "Figure6D_sample_level_heatmap.svg"), bbox_inches="tight")
g.savefig(os.path.join(FIG6_DIR, "Figure6D_sample_level_heatmap.png"), dpi=600, bbox_inches="tight")
plt.close(g.fig)

In [ ]:
# ============================================================
# Save summary tables
# ============================================================

summary = {
    "n_total_PD1_response_samples": len(pd1_data),
    "n_responder": int((pd1_data["Response"] == "Responder").sum()),
    "n_non_responder": int((pd1_data["Response"] == "Non-responder").sum()),
    "main_conclusion": "RLP3/RLP4 program is better interpreted as a hypoxia-associated immune-metabolic state rather than a direct anti-PD-1 response predictor."
}

with open(os.path.join(FIG6_DIR, "Figure6_summary.txt"), "w") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("Figure6 final finished.")
print("Output:", FIG6_DIR)